In [1]:
from datasets import Dataset
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

c:\Programming\Outpeer\capstone_proj\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/final_kk_ru_trad.csv")
df = df[["target_lang", "traditional_translation"]]


In [3]:
raw_dataset = Dataset.from_pandas(df)
raw_dataset = raw_dataset.train_test_split(test_size=0.1)

In [4]:
model_checkpoint = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, src_lang="rus_Cyrl", tgt_lang="kaz_Cyrl")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, use_safetensors=True)

Loading weights: 100%|██████████| 509/509 [00:00<00:00, 8927.75it/s]


In [5]:
def preprocess_function(examples):
    inputs = examples["target_lang"]
    targets = examples["traditional_translation"]
    model_inputs = tokenizer(inputs, max_length=64, truncation=True, padding="max_length")

    labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [6]:
tokenized_datasets = raw_dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 100/100 [00:00<00:00, 7155.68 examples/s]


In [7]:
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [8]:
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/checkpoints",
    learning_rate= 1e-4,
    save_strategy="no",
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    num_train_epochs=8,
    optim="adamw_torch", 
    eval_strategy = "epoch",
    predict_with_generate=True,
    fp16=True,
    logging_steps=5,
    report_to="none",
)

In [9]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model)

In [10]:
trainer = Seq2SeqTrainer(
    model=model,
    args = training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [11]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [12]:
model = prepare_model_for_kbit_training(model)
config = LoraConfig(
    r=16,
    lora_alpha=32,  
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

bin c:\Programming\Outpeer\capstone_proj\venv\Lib\site-packages\bitsandbytes\libbitsandbytes_cuda121.dll
trainable params: 4,718,592 || all params: 619,792,384 || trainable%: 0.7613


In [13]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
Current device: NVIDIA GeForce RTX 3050 Laptop GPU


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,184.335303,11.603323,3.028757
2,181.961597,11.398410,3.011112
3,145.540369,11.156512,2.994593
4,173.720349,10.918694,2.942628
5,138.477905,10.717272,3.054594
6,168.501306,10.564908,3.094835
7,165.460571,10.471157,3.094835
8,133.808521,10.437669,3.080604


TrainOutput(global_step=64, training_loss=153.03460836410522, metrics={'train_runtime': 2689.0045, 'train_samples_per_second': 2.678, 'train_steps_per_second': 0.024, 'total_flos': 988243034112000.0, 'train_loss': 153.03460836410522, 'epoch': 8.0})

In [29]:

# АЛЬТЕРНАТИВНЫЙ МЕТОД СОХРАНЕНИЯ (если save_pretrained не создал adapter_config.json)
from peft import PeftModel
import json
import os

adapter_save_path = "../models/final_adapter_1000"

print("ПРИНУДИТЕЛЬНОЕ ПЕРЕСОХРАНЕНИЕ АДАПТЕРА")
print("=" * 60)

# Шаг 1: Удаляем старую папку если нужно
if os.path.exists(adapter_save_path):
    import shutil
    shutil.rmtree(adapter_save_path)
    print(f"✓ Удалена старая папка: {adapter_save_path}")

# Шаг 2: Пересохраняем адаптер
os.makedirs(adapter_save_path, exist_ok=True)
model.save_pretrained(adapter_save_path)
print(f"✓ Адаптер пересохранен в: {adapter_save_path}")

# Шаг 3: Проверяем файлы
files = os.listdir(adapter_save_path)
print(f"\n✓ Сохраненные файлы ({len(files)}):")
for f in sorted(files):
    file_path = os.path.join(adapter_save_path, f)
    size = os.path.getsize(file_path) / (1024**2)  # в МБ
    print(f"  - {f} ({size:.2f} MB)")

# Шаг 4: Финальная проверка adapter_config.json
adapter_config_path = os.path.join(adapter_save_path, "adapter_config.json")
if os.path.exists(adapter_config_path):
    print(f"\n✓✓✓ adapter_config.json успешно создан!")
    with open(adapter_config_path, 'r') as f:
        config = json.load(f)
    print(f"Содержимое: {json.dumps(config, indent=2)}")
else:
    print(f"\n✗✗✗ adapter_config.json НЕ НАЙДЕН!")
    print(f"Это может быть проблемой при загрузке в Streamlit")


ПРИНУДИТЕЛЬНОЕ ПЕРЕСОХРАНЕНИЕ АДАПТЕРА


Writing model shards: 100%|██████████| 1/1 [00:24<00:00, 24.12s/it]

✓ Адаптер пересохранен в: ../models/final_adapter_1000

✓ Сохраненные файлы (3):
  - config.json (0.00 MB)
  - generation_config.json (0.00 MB)
  - model.safetensors (2346.38 MB)

✗✗✗ adapter_config.json НЕ НАЙДЕН!
Это может быть проблемой при загрузке в Streamlit


In [15]:
from transformers import pipeline
import os

# 1. Пути и загрузка (поднимаемся из notebooks в корень)
current_dir = os.path.dirname(os.getcwd())
model_path = os.path.join(current_dir, "models", "checkpoints", "checkpoint-565")

print(f"Загрузка из: {model_path}")

# Загружаем токенизатор с указанием исходного языка
tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang="rus_Cyrl")
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Переносим на GPU если есть
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Список фраз для проверки
test_phrases = [
    "Вы хотите этот напиток?", 
    "Этот закон был принят в Казахстане.", 
    "Я и мой брат пошли домой.", 
    "Информация о налогах."
]

print("\n--- РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ ---")

for phrase in test_phrases:
    # Токенизация (явно указываем тензоры для девайса)
    inputs = tokenizer(phrase, return_tensors="pt").to(device)

    kaz_lang_id = tokenizer.convert_tokens_to_ids("kaz_Cyrl")
    
    # Генерация (принудительно ставим токен начала казахского языка)
    result = model.generate(
        **inputs, 
        forced_bos_token_id=kaz_lang_id, 
        max_length=64,
        num_beams=5, # добавим beam search для лучшего качества
        early_stopping=True
    )
    
    # Декодирование
    output = tokenizer.batch_decode(result, skip_special_tokens=True)[0]
    
    print(f"RU: {phrase}")
    print(f"KZ: {output}")
    print("-" * 30)

Загрузка из: c:\Programming\Outpeer\capstone_proj\models\checkpoints\checkpoint-565


Loading weights: 100%|██████████| 509/509 [00:00<00:00, 4349.60it/s]



--- РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ ---
RU: Вы хотите этот напиток?
KZ: Бұл ішуді қалайсыз ба?
------------------------------
RU: Этот закон был принят в Казахстане.
KZ: Бұл заң Қазақстанда қабылданды.
------------------------------
RU: Я и мой брат пошли домой.
KZ: Ағам екеуміз үйге бардық.
------------------------------
RU: Информация о налогах.
KZ: Салық туралы ақпарат.
------------------------------


In [16]:
test_heavy_phrases = [
    "В соответствии с вышеуказанным приказом, работа была начата.",
    "По причине отсутствия времени мы не смогли прийти.",
    "Данный вопрос рассматривается в рамках текущей встречи.",
    "С целью повышения эффективности обучения был создан новый курс."
]

print("--- ТЕСТ НА СЛОЖНЫЕ КАЛЬКИ ---")

for phrase in test_heavy_phrases:
    inputs = tokenizer(phrase, return_tensors="pt").to(device)
    
    # Используем те же параметры генерации
    result = model.generate(
        **inputs, 
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("kaz_Cyrl"), 
        max_length=128,
        num_beams=5,
        early_stopping=True
    )
    
    output = tokenizer.batch_decode(result, skip_special_tokens=True)[0]
    
    print(f"RU: {phrase}")
    print(f"KZ: {output}")
    print("-" * 30)

--- ТЕСТ НА СЛОЖНЫЕ КАЛЬКИ ---
RU: В соответствии с вышеуказанным приказом, работа была начата.
KZ: Жоғарыда көрсетілген бұйрыққа сәйкес жұмыс басталды.
------------------------------
RU: По причине отсутствия времени мы не смогли прийти.
KZ: Уақыт жетіспегендіктен келе алмадық.
------------------------------
RU: Данный вопрос рассматривается в рамках текущей встречи.
KZ: Бұл мәселе кездесу аясында қаралады.
------------------------------
RU: С целью повышения эффективности обучения был создан новый курс.
KZ: Оқытудың тиімділігін арттыру мақсатында жаңа курс құрылды.
------------------------------


In [17]:
# Тестируем финальную версию
test_sentences = [
    "В соответствии с вышеуказанным приказом, работа была начата.",
    "По причине отсутствия времени мы не смогли прийти.",
    "Я и мой брат пошли домой."
]

print("--- ТЕСТ МОДЕЛИ (1000 СТРОК + LoRA) ---")

for text in test_sentences:
    inputs = tokenizer(text, return_tensors="pt").to("cuda") # Теперь точно на CUDA!
    
    with torch.cuda.amp.autocast(): # Используем смешанную точность для скорости
        generated_tokens = model.generate(
            **inputs, 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("kaz_Cyrl"),
            max_length=128,
            num_beams=5
        )
    
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    print(f"RU: {text}")
    print(f"KZ: {result}")
    print("-" * 30)

--- ТЕСТ МОДЕЛИ (1000 СТРОК + LoRA) ---


C:\Users\Techno\AppData\Local\Temp\ipykernel_22800\4116283634.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # Используем смешанную точность для скорости


RU: В соответствии с вышеуказанным приказом, работа была начата.
KZ: Жоғарыда көрсетілген бұйрыққа сәйкес жұмыс басталды.
------------------------------
RU: По причине отсутствия времени мы не смогли прийти.
KZ: Уақыт жетіспегендіктен келе алмадық.
------------------------------
RU: Я и мой брат пошли домой.
KZ: Ағам екеуміз үйге бардық.
------------------------------


In [18]:
final_boss_sentences = [
    "Человек, пришедший вчера, принес важные документы.",
    "В связи с изменением графика работы, просим вас ознакомиться с новым расписанием.",
    "Добро пожаловать в наш город, надеемся на плодотворное сотрудничество."
]

print("--- ФИНАЛЬНЫЙ СТРЕСС-ТЕСТ ---")

for text in final_boss_sentences:
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    with torch.cuda.amp.autocast():
        generated_tokens = model.generate(
            **inputs, 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("kaz_Cyrl"),
            max_length=128,
            num_beams=5,
            no_repeat_ngram_size=2
        )
    
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    print(f"RU: {text}")
    print(f"KZ: {result}")
    print("-" * 30)

--- ФИНАЛЬНЫЙ СТРЕСС-ТЕСТ ---


C:\Users\Techno\AppData\Local\Temp\ipykernel_22800\2561028408.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


RU: Человек, пришедший вчера, принес важные документы.
KZ: Келесі күні келген адам маңызды құжаттарды алып келді.
------------------------------
RU: В связи с изменением графика работы, просим вас ознакомиться с новым расписанием.
KZ: Жұмыс кестесі өзгергендіктен, жаңа кестемен танысуыңызды сұраймыз.
------------------------------
RU: Добро пожаловать в наш город, надеемся на плодотворное сотрудничество.
KZ: Қаламызға қош келдіңіздер, жемісті ынтымақтастыққа үміттенеміз.
------------------------------
